# Direct Forecast — Safe Lag v3

## Cải thiện so với v2 (forecasting_safe_lag.ipynb)

| Vấn đề v2 | Giải pháp v3 |
|---|---|
| `rolling_mean_28` = shift(1): Aug 31 chỉ có 12/28 ngày thật → distribution shift | Xóa; dùng `rolling_mean_28_safe` = shift(16) → 28 ngày thật |
| `rolling_std_7` = shift(1): shrinking window → bias | Xóa; dùng `rolling_std_28_safe` = shift(16) → consistent |
| 2 extra lag = chỉ lag_16 | Thêm lag_21, lag_35 → weekly pattern signal |

## Feature safety (Aug 16–31)

| Feature | Lookback | Status |
|---|---|---|
| lag_14 | cần Aug 2–17 | safe 14/16 ngày; NaN Aug 30-31 (model handles) |
| lag_16 | cần Aug 1–15 | **100% SAFE** ✓ |
| lag_21 | cần Jul 26–Aug 10 | **100% SAFE** ✓ |
| lag_28 | cần Jul 19–Aug 3 | **100% SAFE** ✓ |
| lag_35 | cần Jul 12–Jul 27 | **100% SAFE** ✓ |
| lag_364 | cần Aug 2016 | **100% SAFE** ✓ |
| rolling_mean_30 | shift(16).rolling(30) | **100% SAFE** ✓ |
| rolling_mean_28_safe | shift(16).rolling(28) | **100% SAFE** ✓ |
| rolling_std_28_safe | shift(16).rolling(28).std | **100% SAFE** ✓ |

## 1. Load Data & Verify Prerequisites

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [_here] + list(_here.parents) if (p / 'config.py').exists()),
    _here
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import PROJECT_ROOT, DATA_DIR, RAW_DIR, PROCESSED_DIR, SPLITS_DIR

MODELS_DIR    = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
SPLITS_V3     = PROCESSED_DIR / 'splits_safe_lag_v3'

assert SPLITS_V3.exists(), f'Chạy feature_safe_lag_v3.ipynb trước!'

for model_f, nb in [
    (NOTEBOOKS_DIR / '08_forecasting' / 'xgb_safe_lag_v2_model.pkl', 'XGBoost_training_safe_lag_v2.ipynb'),
    (NOTEBOOKS_DIR / '08_forecasting' / 'lgb_safe_lag_v2_model.pkl', 'LightGBM_training_safe_lag_v2.ipynb'),
]:
    assert model_f.exists(), f'Model không tồn tại: {model_f}\nChạy {nb} trước!'

print('All prerequisites ✓')

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_log_error, mean_squared_error, mean_absolute_error
from category_encoders import TargetEncoder

# Kaggle test set (2017-08-16 → 2017-08-31)
test_kaggle = pd.read_csv(PROCESSED_DIR / 'cleaned' / 'test_cleaned.csv')

# Full historical training data (2013-01-01 → 2017-08-15)
train_raw = pd.read_csv(PROCESSED_DIR / 'cleaned' / 'train_cleaned.csv')

oil_df      = pd.read_csv(RAW_DIR / 'oil.csv')
holidays_df = pd.read_csv(RAW_DIR / 'holidays_events.csv')
stores_df   = pd.read_csv(RAW_DIR / 'stores.csv')

# Authoritative column order từ v3 splits (47 cols)
ref_cols = pd.read_csv(SPLITS_V3 / 'train_features.csv', nrows=0).columns.tolist()

print('Data loaded:')
for name, df in [('test_kaggle', test_kaggle), ('train_raw', train_raw)]:
    print(f'  {name:<12}: {df.shape}')
print(f'  ref_cols    : {len(ref_cols)} features  (expected 47)')
assert len(ref_cols) == 47, f'Expected 47 ref_cols, got {len(ref_cols)}'

v3_check = ['lag_16', 'lag_21', 'lag_35', 'rolling_mean_30', 'rolling_mean_28_safe', 'rolling_std_28_safe']
removed_check = ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'rolling_mean_28', 'rolling_std_7']
for col in v3_check:
    print(f'  {col:<25}: {"in ref_cols ✓" if col in ref_cols else "MISSING ✗"}')
for col in removed_check:
    print(f'  {col:<25}: {"absent ✓" if col not in ref_cols else "STILL IN ref_cols ✗"}')

## 2. Feature Engineering (Direct — no iterative loop)

Tất cả features tính từ historical data thật. Không có vòng lặp dự báo.

In [ ]:
X_kaggle = test_kaggle.copy()
X_kaggle['date'] = pd.to_datetime(X_kaggle['date'])
train_raw['date'] = pd.to_datetime(train_raw['date'])

# Calendar features
dt = X_kaggle['date'].dt
X_kaggle['year']           = dt.year
X_kaggle['month']          = dt.month
X_kaggle['day']            = dt.day
X_kaggle['day_of_week']    = dt.dayofweek
X_kaggle['week_of_year']   = dt.isocalendar().week.astype(int)
X_kaggle['quarter']        = dt.quarter
X_kaggle['is_weekend']     = (dt.dayofweek >= 5).astype(int)
X_kaggle['is_month_start'] = dt.is_month_start.astype(int)
X_kaggle['is_month_end']   = dt.is_month_end.astype(int)
X_kaggle['is_payday']      = ((dt.day == 15) | dt.is_month_end).astype(int)

X_kaggle = X_kaggle.merge(
    stores_df[['store_nbr', 'type', 'city', 'state', 'cluster']],
    on='store_nbr', how='left'
)
assert X_kaggle.shape[0] == len(test_kaggle), 'Store merge changed row count'
print(f'After calendar + store: {X_kaggle.shape}')

In [ ]:
# ── Lag features via merge approach ─────────────────────────────────────────
# lag_N tại ngày d = sales tại d-N
# Nếu d-N ≤ 2017-08-15 → lấy từ train_raw (real historical data)
#
# Safety matrix (Aug 16-31):
#   lag_14: Aug 16 cần Aug 2 ✓, Aug 29 cần Aug 15 ✓, Aug 30 cần Aug 16 ✗ → NaN 2 ngày
#   lag_16: Aug 16 cần Jul 31 ✓, Aug 31 cần Aug 15 ✓ → 100% SAFE
#   lag_21: Aug 16 cần Jul 26 ✓, Aug 31 cần Aug 10 ✓ → 100% SAFE
#   lag_28: Aug 16 cần Jul 19 ✓, Aug 31 cần Aug 3 ✓  → 100% SAFE
#   lag_35: Aug 16 cần Jul 12 ✓, Aug 31 cần Jul 27 ✓ → 100% SAFE
#   lag_364: Aug 16 cần Aug 17/2016 ✓                → 100% SAFE

for lag in [14, 16, 21, 28, 35, 364]:
    lag_ref = train_raw[['date', 'store_nbr', 'family', 'sales']].copy()
    lag_ref['date'] = lag_ref['date'] + pd.Timedelta(days=lag)
    lag_ref = lag_ref.rename(columns={'sales': f'lag_{lag}'})
    X_kaggle = X_kaggle.merge(
        lag_ref[['date', 'store_nbr', 'family', f'lag_{lag}']],
        on=['date', 'store_nbr', 'family'], how='left'
    )

assert X_kaggle.shape[0] == len(test_kaggle), 'Lag merge changed row count'

print('Lag feature NaN counts:')
for lag in [14, 16, 21, 28, 35, 364]:
    n    = X_kaggle[f'lag_{lag}'].isnull().sum()
    days = n // 1782 if n > 0 else 0  # 1782 rows per day (54 stores × 33 families)
    status = 'SAFE ✓ 0 NaN' if n == 0 else f'NaN {days}/16 ngày (model native NaN)'
    print(f'  lag_{lag:<4}: {n:>5} NaN — {status}')

# Critical: các lag hoàn toàn safe phải có 0 NaN
for lag in [16, 21, 28, 35, 364]:
    n = X_kaggle[f'lag_{lag}'].isnull().sum()
    assert n == 0, f'FAIL: lag_{lag} có {n} NaN — unexpected!'
print('\nAssertion passed: lag_16/21/28/35/364 all zero NaN ✓')

In [ ]:
# ── Rolling features — CHỈ dùng shift(16)-based (100% safe) ──────────────────
# Combined-append approach:
#   1. Concat recent train history + test rows (NaN sales)
#   2. Compute rolling với shift(16) → tất cả test rows' shifted values = real train dates
#   3. Extract test rows
#
# shift(16).rolling(W) tại Aug 16-31: shift picks July 31 – Aug 15 (all real) → 0 NaN guaranteed

test_start = X_kaggle['date'].min()  # 2017-08-16

# Cần đủ history cho shift(16).rolling(30): Aug 16 cần July 2 (tối sớm nhất)
# Aug 16 - 16 - 30 + 1 = July 2 → cutoff = July 1 (an toàn)
cutoff = test_start - pd.Timedelta(days=46)

train_recent = train_raw.loc[
    train_raw['date'] >= cutoff,
    ['date', 'store_nbr', 'family', 'sales']
].copy()

test_rows = X_kaggle[['date', 'store_nbr', 'family']].assign(sales=np.nan)

combined = pd.concat([train_recent, test_rows]).sort_values(
    ['store_nbr', 'family', 'date']
).reset_index(drop=True)

print(f'Combined: {combined.shape}  (train_recent={len(train_recent):,}, test_rows={len(test_rows):,})')

grp = combined.groupby(['store_nbr', 'family'], sort=False)['sales']

# rolling_mean_30: shift(16).rolling(30) → lookback d-45 đến d-16 ✓
combined['rolling_mean_30'] = grp.transform(
    lambda x: x.shift(16).rolling(30, min_periods=1).mean()
)

# rolling_mean_28_safe: shift(16).rolling(28) → lookback d-43 đến d-16 ✓
combined['rolling_mean_28_safe'] = grp.transform(
    lambda x: x.shift(16).rolling(28, min_periods=1).mean()
)

# rolling_std_28_safe: shift(16).rolling(28).std → same lookback ✓
combined['rolling_std_28_safe'] = grp.transform(
    lambda x: x.shift(16).rolling(28, min_periods=2).std()
)

# Extract test rows
test_mask    = combined['date'].isin(X_kaggle['date'].unique())
roll_cols    = ['date', 'store_nbr', 'family',
                'rolling_mean_30', 'rolling_mean_28_safe', 'rolling_std_28_safe']
test_rolling = combined.loc[test_mask, roll_cols].reset_index(drop=True)

X_kaggle = X_kaggle.merge(test_rolling, on=['date', 'store_nbr', 'family'], how='left')
assert X_kaggle.shape[0] == len(test_kaggle), 'Rolling merge changed row count'

del combined, train_recent, test_rows, test_rolling, grp

print('Rolling feature NaN counts:')
for col in ['rolling_mean_30', 'rolling_mean_28_safe', 'rolling_std_28_safe']:
    n = X_kaggle[col].isnull().sum()
    status = 'SAFE ✓ 0 NaN' if n == 0 else f'FAIL ✗ {n} NaN'
    print(f'  {col:<25}: {n:>5} NaN — {status}')

for col in ['rolling_mean_30', 'rolling_mean_28_safe', 'rolling_std_28_safe']:
    assert X_kaggle[col].isnull().sum() == 0, f'FAIL: {col} has NaN!'
print('\nAll rolling assertions passed ✓')

In [ ]:
# ── Holiday features ──────────────────────────────────────────────────────────
holidays = holidays_df.copy()
holidays['date'] = pd.to_datetime(holidays['date'])
valid_holidays   = holidays[holidays['transferred'] == False].copy()

type_mapping = {'Holiday': 1, 'Event': 2, 'Additional': 3,
                'Transfer': 4, 'Bridge': 5, 'Work Day': 0}
valid_holidays['holiday_type_encoded'] = valid_holidays['type'].map(type_mapping).fillna(0).astype(int)
valid_holidays['is_transferred']       = valid_holidays['transferred'].astype(int)
valid_holidays['is_carnaval']          = (
    valid_holidays['description'].str.contains('Carnaval', case=False, na=False).astype(int)
)

for col in ['is_national_holiday', 'is_regional_holiday', 'is_local_holiday',
            'is_transferred_holiday', 'holiday_type', 'is_carnaval_feature']:
    X_kaggle[col] = 0

scope_col = {'National': 'is_national_holiday',
             'Regional': 'is_regional_holiday',
             'Local':    'is_local_holiday'}

for _, row in valid_holidays.iterrows():
    h_date, h_locale, h_name = row['date'], row['locale'], row['locale_name']
    if h_locale == 'National':
        mask = X_kaggle['date'] == h_date
    elif h_locale == 'Regional':
        mask = (X_kaggle['date'] == h_date) & (X_kaggle['state'] == h_name)
    elif h_locale == 'Local':
        mask = (X_kaggle['date'] == h_date) & (X_kaggle['city'] == h_name)
    else:
        continue
    if not mask.any():
        continue
    X_kaggle.loc[mask, scope_col[h_locale]]      = 1
    X_kaggle.loc[mask, 'is_transferred_holiday'] = row['is_transferred']
    X_kaggle.loc[mask, 'holiday_type']           = row['holiday_type_encoded']
    X_kaggle.loc[mask, 'is_carnaval_feature']    = row['is_carnaval']

unique_hol_dates = valid_holidays['date'].drop_duplicates().sort_values().values

def _days_to_next(d):
    future = unique_hol_dates[unique_hol_dates > d]
    return int((future[0] - d) / np.timedelta64(1, 'D')) if len(future) > 0 else -1

def _days_after_last(d):
    past = unique_hol_dates[unique_hol_dates < d]
    return int((d - past[-1]) / np.timedelta64(1, 'D')) if len(past) > 0 else -1

uq_dates = X_kaggle[['date']].drop_duplicates().copy()
uq_dates['days_to_next_holiday']    = uq_dates['date'].apply(_days_to_next)
uq_dates['days_after_last_holiday'] = uq_dates['date'].apply(_days_after_last)
X_kaggle = X_kaggle.merge(uq_dates, on='date', how='left')

X_kaggle['is_earthquake_period'] = 0

assert X_kaggle.shape[0] == len(test_kaggle), 'Holiday merge changed row count'
print(f'National holidays : {X_kaggle["is_national_holiday"].sum()}')
print(f'Regional holidays : {X_kaggle["is_regional_holiday"].sum()}')
print(f'Local holidays    : {X_kaggle["is_local_holiday"].sum()}')

In [ ]:
# ── Oil price features ────────────────────────────────────────────────────────
oil = oil_df.copy().rename(columns={'dcoilwtico': 'oil_price'})
oil['date'] = pd.to_datetime(oil['date'])
oil = oil[['date', 'oil_price']].sort_values('date').reset_index(drop=True)

max_date   = max(oil['date'].max(), X_kaggle['date'].max())
full_range = pd.DataFrame({'date': pd.date_range(oil['date'].min(), max_date, freq='D')})
oil = full_range.merge(oil, on='date', how='left')
oil['oil_price'] = oil['oil_price'].ffill().bfill()

oil['oil_price_lag_7']           = oil['oil_price'].shift(7)
oil['oil_price_lag_14']          = oil['oil_price'].shift(14)
oil['oil_price_rolling_mean_28'] = oil['oil_price'].shift(1).rolling(28, min_periods=7).mean()
oil['oil_price_change_pct']      = oil['oil_price'].pct_change(periods=7)

oil_cols = ['date', 'oil_price', 'oil_price_lag_7', 'oil_price_lag_14',
            'oil_price_rolling_mean_28', 'oil_price_change_pct']
X_kaggle = X_kaggle.merge(oil[oil_cols], on='date', how='left')

assert X_kaggle.shape[0] == len(test_kaggle), 'Oil merge changed row count'
print(f'Oil NaN total     : {X_kaggle[oil_cols[1:]].isnull().sum().sum()}')
print(f'Oil price range   : {X_kaggle["oil_price"].min():.2f} – {X_kaggle["oil_price"].max():.2f}')

In [ ]:
# ── Store encoding + target encoding ─────────────────────────────────────────
type_map = {t: i for i, t in enumerate(sorted(stores_df['type'].unique()))}
X_kaggle['store_type_enc']     = X_kaggle['type'].map(type_map)
X_kaggle['store_type_encoded'] = X_kaggle['type'].map(type_map)

city_freq_map  = stores_df['city'].value_counts(normalize=True).to_dict()
state_freq_map = stores_df['state'].value_counts(normalize=True).to_dict()
X_kaggle['city_freq']  = X_kaggle['city'].map(city_freq_map)
X_kaggle['state_freq'] = X_kaggle['state'].map(state_freq_map)

X_kaggle['store_family'] = X_kaggle['store_nbr'].astype(str) + '_' + X_kaggle['family']

train_raw['store_family'] = train_raw['store_nbr'].astype(str) + '_' + train_raw['family']
te_enc = TargetEncoder(cols=['store_family'], smoothing=10)
te_enc.fit(train_raw[['store_family']], train_raw['sales'])
X_kaggle['store_family_te'] = te_enc.transform(X_kaggle[['store_family']])['store_family']

X_kaggle['date'] = X_kaggle['date'].dt.strftime('%Y-%m-%d')

kaggle_ids = test_kaggle['id'].reset_index(drop=True)
X_kaggle   = X_kaggle.drop(columns=['id'], errors='ignore')

missing_cols = set(ref_cols) - set(X_kaggle.columns)
extra_cols   = set(X_kaggle.columns) - set(ref_cols)
assert not missing_cols, f'Missing columns: {missing_cols}'
assert not extra_cols,   f'Extra columns (not in ref_cols): {extra_cols}'

X_kaggle = X_kaggle[ref_cols]
assert X_kaggle.shape == (len(test_kaggle), 47), f'Final shape mismatch: {X_kaggle.shape}'

print(f'X_kaggle final: {X_kaggle.shape}  (expected {len(test_kaggle)} × 47)')
nan_nonzero = X_kaggle.isnull().sum()
nan_nonzero = nan_nonzero[nan_nonzero > 0]
if len(nan_nonzero):
    print(f'\nColumns với NaN (model xử lý natively):')
    print(nan_nonzero.to_string())
else:
    print('\nNo missing values — perfect!')

## 2b. Verify All Safe Features = 0 NaN

In [ ]:
print('=== VERIFICATION: All safe features cho Kaggle test (Aug 16-31) ===\n')
safe_features = ['lag_16', 'lag_21', 'lag_28', 'lag_35', 'lag_364',
                 'rolling_mean_30', 'rolling_mean_28_safe', 'rolling_std_28_safe']
for col in safe_features:
    n = X_kaggle[col].isnull().sum()
    status = 'PASS ✓ (0 NaN — 100% real historical data)' if n == 0 else f'FAIL ✗ ({n} NaN)'
    print(f'  {col:<25}: {status}')
    assert n == 0, f'BUG: {col} has {n} NaN!'

print()
print('Sample (5 rows, key safe lag features):')
key_cols = ['date', 'store_nbr', 'lag_14', 'lag_16', 'lag_21', 'lag_35',
            'rolling_mean_28_safe', 'rolling_std_28_safe']
print(X_kaggle[key_cols].head(5).to_string())

## 3. Encode Categorical Features

In [ ]:
object_cols = X_kaggle.select_dtypes(include=['object']).columns.tolist()

X_train_obj = pd.read_csv(SPLITS_V3 / 'train_features.csv', usecols=lambda c: c in object_cols)
X_val_obj   = pd.read_csv(SPLITS_V3 / 'val_features.csv',   usecols=lambda c: c in object_cols)
X_test_obj  = pd.read_csv(SPLITS_V3 / 'test_features.csv',  usecols=lambda c: c in object_cols)

label_encoders = {}
for col in object_cols:
    le    = LabelEncoder()
    parts = [X_train_obj[col], X_val_obj[col]]
    if col in X_test_obj.columns:
        parts.append(X_test_obj[col])
    parts.append(X_kaggle[col])
    combined = pd.concat(parts, ignore_index=True).astype(str)
    le.fit(combined)
    X_kaggle[col] = le.transform(X_kaggle[col].astype(str)).astype(np.int32)
    label_encoders[col] = le

del X_train_obj, X_val_obj, X_test_obj

remaining = X_kaggle.select_dtypes(include=['object']).columns.tolist()
assert not remaining, f'Still has object columns: {remaining}'
print(f'Encoded {len(object_cols)} cols: {object_cols}')
print(f'X_kaggle final: {X_kaggle.shape}')

## 4. Load Models & Direct Predict

**Direct forecast**: `model.predict()` **một lần** cho toàn bộ 28,512 rows — không có loop.

In [ ]:
lgb_model = joblib.load(NOTEBOOKS_DIR / '08_forecasting' / 'lgb_safe_lag_v2_model.pkl')
xgb_model = joblib.load(NOTEBOOKS_DIR / '08_forecasting' / 'xgb_safe_lag_v2_model.pkl')

lgb_w, xgb_w = 0.3, 0.7

assert lgb_model.n_features_ == 47, f'LGB expects 47 features, got {lgb_model.n_features_}'
assert xgb_model.n_features_in_ == 47, f'XGB expects 47 features, got {xgb_model.n_features_in_}'

print(f'Models loaded.  LGB weight: {lgb_w}  |  XGB weight: {xgb_w}')
print(f'Running DIRECT predict on {len(X_kaggle):,} rows (no loop)...')

lgb_pred = lgb_model.predict(X_kaggle)
xgb_pred = xgb_model.predict(X_kaggle)

final_pred_log = lgb_w * lgb_pred + xgb_w * xgb_pred
final_pred     = np.maximum(np.expm1(final_pred_log), 0)

print(f'\nPredictions:')
print(f'  min  : {final_pred.min():.4f}')
print(f'  max  : {final_pred.max():.2f}')
print(f'  mean : {final_pred.mean():.2f}')
print(f'  NaN  : {np.isnan(final_pred).sum()}')

## 5. Evaluate trên Local Test Set (Aug 01–15)

In [ ]:
X_test_local = pd.read_csv(SPLITS_V3 / 'test_features.csv')
y_test_local = pd.read_csv(SPLITS_V3 / 'test_target.csv')

# Encode categoricals
for col in object_cols:
    if col in X_test_local.columns:
        le = label_encoders[col]
        X_test_local[col] = X_test_local[col].astype(str).apply(
            lambda x: le.transform([x])[0] if x in le.classes_ else -1
        ).astype(np.int32)

lgb_loc    = lgb_model.predict(X_test_local)
xgb_loc    = xgb_model.predict(X_test_local)
blend_log  = lgb_w * lgb_loc + xgb_w * xgb_loc
blend_pred = np.maximum(np.expm1(blend_log), 0)

y_true = np.expm1(y_test_local['sales_log'].to_numpy())

rmsle    = np.sqrt(mean_squared_log_error(np.clip(y_true, 0, None), np.clip(blend_pred, 0, None)))
rmse     = np.sqrt(mean_squared_error(y_true, blend_pred))
mae      = mean_absolute_error(y_true, blend_pred)
rmse_log = np.sqrt(mean_squared_error(y_test_local['sales_log'].to_numpy(), blend_log))

print(f'Ensemble LGB{lgb_w}–XGB{xgb_w} | Local Test (Aug 01–15) | v3 safe lags')
print('=' * 60)
print(f'  RMSLE    : {rmsle:.6f}')
print(f'  RMSE     : {rmse:.4f}')
print(f'  MAE      : {mae:.4f}')
print(f'  RMSE_log : {rmse_log:.6f}')
print('=' * 60)
print(f'\nBaseline comparisons (local test Aug 01-15):')
print(f'  v2 safe lag (old rolling features) : 0.?????? (xem forecasting_safe_lag.ipynb)')
print(f'  toxic lags direct (original)        : 0.370354')
print(f'  v3 safe lag (this)                  : {rmsle:.6f}')

## 6. Tạo Submission File

In [ ]:
submission = pd.DataFrame({'id': kaggle_ids, 'sales': final_pred})

assert len(submission) == len(test_kaggle),     'Row count mismatch'
assert submission['sales'].isnull().sum() == 0, 'NaN in sales'
assert (submission['sales'] < 0).sum() == 0,   'Negative sales'

out_dir  = NOTEBOOKS_DIR / '08_ensemble'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'submission_safe_lag_v2.csv'
submission.to_csv(out_path, index=False)

print(f'Submission saved: {out_path}')
print(f'Shape           : {submission.shape}')
print(f'Sales range     : [{submission["sales"].min():.4f}, {submission["sales"].max():.2f}]')
print(f'Sales mean      : {submission["sales"].mean():.2f}')
print()
print(submission.head(10).to_string())

## Summary

| Pipeline | Rolling features | Lag features | Expected Kaggle score |
|---|---|---|---|
| v2 (cũ) | shift(1) → distribution shift Aug 17-31 | lag_16 only | 1.05175 |
| **v3 (này)** | **shift(16) → consistent** | **lag_16/21/35** | **?** |

**Thay đổi chính:**
- `rolling_mean_28` → `rolling_mean_28_safe`: cùng window size (28 ngày) nhưng shift(16) thay vì shift(1)
- `rolling_std_7` → `rolling_std_28_safe`: window lớn hơn, stable, consistent giữa train và test
- Thêm `lag_21`, `lag_35`: bắt weekly pattern tốt hơn (3 tuần, 5 tuần trước)

**Zero distribution shift**: mọi rolling feature trong training và forecasting đều được tính với cùng shift(16) → model thấy exact cùng distribution như lúc training.